# 01 - Data Preparation: MD Trajectory
This notebook generates a Molecular Dynamics (MD) trajectory of a Copper supercell using the ASE EMT calculator. The trajectory provides correlated structures with non-zero forces, necessary for benchmarking force-matching interatomic potentials.

In [1]:
import os
import numpy as np
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet
from ase.io import write
from ase.calculators.singlepoint import SinglePointCalculator
from ase import units
from tqdm.auto import tqdm

os.makedirs('../data', exist_ok=True)
np.random.seed(42)

# 1. Setup Initial Structure and Calculator
print("Setting up Cu supercell...")
atoms = bulk('Cu', 'fcc', a=3.6) * (2, 2, 2)
atoms.calc = EMT()

# 2. Initialize MD (Temperature = 300K)
MaxwellBoltzmannDistribution(atoms, temperature_K=300)
dyn = VelocityVerlet(atoms, 1.0 * units.fs)

# 3. Run MD and collect 1400 samples
n_samples = 1400
dataset = []

print(f"Running MD simulation to generate {n_samples} frames...")
for _ in tqdm(range(n_samples), desc="MD Steps"):
    # Run a few steps between sampling to decorrelate slightly
    dyn.run(10)
    
    # Extract current state
    new_atoms = atoms.copy()
    energy = atoms.get_potential_energy()
    forces = atoms.get_forces()
    
    # Store energy and forces permanently via SinglePointCalculator
    calc = SinglePointCalculator(new_atoms, energy=energy, forces=forces)
    new_atoms.calc = calc
    
    dataset.append(new_atoms)

Setting up Cu supercell...
Running MD simulation to generate 1400 frames...


MD Steps:   0%|          | 0/1400 [00:00<?, ?it/s]

In [2]:
import random
random.seed(42)

# Shuffle the trajectory so train/val/test are drawn from the entire distribution
random.shuffle(dataset)

# Experiment A Data Subsets (10%, 40%, 70%, 100%)
train_10_data  = dataset[:100]
train_40_data  = dataset[:400]
train_70_data  = dataset[:700]
train_100_data = dataset[:1000]

val_data   = dataset[1000:1200]
test_data  = dataset[1200:1400]

write('../data/train_10.extxyz', train_10_data)
write('../data/train_40.extxyz', train_40_data)
write('../data/train_70.extxyz', train_70_data)
write('../data/train_100.extxyz', train_100_data)
write('../data/val.extxyz',   val_data)
write('../data/test.extxyz',  test_data)

print('Data Splitting Complete!')
print(f'Train (10%): {len(train_10_data)} | Train (40%): {len(train_40_data)} | Train (70%): {len(train_70_data)} | Train (100%): {len(train_100_data)}')
print(f'Val: {len(val_data)} | Test: {len(test_data)}')


Data Splitting Complete!
Train (10%): 100 | Train (40%): 400 | Train (70%): 700 | Train (100%): 1000
Val: 200 | Test: 200
